In [7]:
import time
import os
import pandas as pd
import nltk as nltk

os.environ["NLTK_ALLOW_PROXIED_URLOPEN"] = "1"

nltk.download("vader_lexicon")
nltk.download("stopwords")

filtered_csv_file = "../../data/Hotel_Reviews_Filtered.csv"

df = pd.read_csv(filtered_csv_file)

df.head()

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/cai/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package stopwords to /home/cai/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,Hotel_Address,Average_Score,Hotel_Name,Reviewer_Nationality,Negative_Review,Total_Number_of_Reviews,Positive_Review,Reviewer_Score,Tags,Calc_Average_Score,Average_Score_Difference,Leisure_trip,Couple,Solo_traveler,Business_trip,Group,Family_with_young_children,Family_with_older_children,With_a_pet
0,"Amsterdam, Netherlands",7.8,Hotel Arena,Russia,I am so angry that i made this post available...,405,Only the park outside of the hotel was beauti...,2.9,"Leisure trip, Couple, Duplex Double Room, Sta...",7.8,-0.1,1,1,0,0,0,0,0,0
1,"Amsterdam, Netherlands",7.8,Hotel Arena,Ireland,No Negative,405,No real complaints the hotel was great great ...,7.5,"Leisure trip, Couple, Duplex Double Room, Sta...",7.8,-0.1,1,1,0,0,0,0,0,0
2,"Amsterdam, Netherlands",7.8,Hotel Arena,Australia,Rooms are nice but for elderly a bit difficul...,405,Location was good and staff were ok It is cut...,7.1,"Leisure trip, Family with young children, Dup...",7.8,-0.1,1,0,0,0,0,1,0,0
3,"Amsterdam, Netherlands",7.8,Hotel Arena,United Kingdom,My room was dirty and I was afraid to walk ba...,405,Great location in nice surroundings the bar a...,3.8,"Leisure trip, Solo traveler, Duplex Double Ro...",7.8,-0.1,1,0,0,0,0,0,0,0
4,"Amsterdam, Netherlands",7.8,Hotel Arena,New Zealand,You When I booked with your company on line y...,405,Amazing location and building Romantic setting,6.7,"Leisure trip, Couple, Suite, Stayed 2 nights,...",7.8,-0.1,1,1,0,0,0,0,0,0


In [8]:
# remove the stop words
from nltk.corpus import stopwords

start  = time.time()

cache = set(stopwords.words("english"))

def remove_stopwords(review):
    text = " ".join([word for word in review.split() if word not in cache])
    return text

df.Negative_Review = df.Negative_Review.apply(remove_stopwords)
df.Positive_Review = df.Positive_Review.apply(remove_stopwords)


In [9]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

vader_sentiment = SentimentIntensityAnalyzer()

def calc_sentiment(review):
    if review == "No Negative" or review == "No Positive":
        return 0
    return vader_sentiment.polarity_scores(review)["compound"]

print("Calculating sentiment columns for both positive and negative reviews")

start = time.time()

df["Negative_Sentiment"] = df.Negative_Review.apply(calc_sentiment)
df["Positive_Sentiment"] = df.Positive_Review.apply(calc_sentiment)

end = time.time()

print(f"Calculating sentiment took {round(end - start, 2)} seconds")


Calculating sentiment columns for both positive and negative reviews
Calculating sentiment took 46.97 seconds


In [11]:
df = df.sort_values(by=["Negative_Sentiment"], ascending=True)
print(df[["Negative_Review", "Negative_Sentiment"]])

df = df.sort_values(by=["Positive_Sentiment"], ascending=True)
print(df[["Positive_Review", "Positive_Sentiment"]])

                                          Negative_Review  Negative_Sentiment
186584  So bad experience memories I hotel The first n...             -0.9920
129503  First charged twice room booked booking second...             -0.9896
307286  The staff Had bad experience even booking Janu...             -0.9889
201953  Everything DO NOT STAY AT THIS HOTEL I never i...             -0.9886
452092  No WLAN room Incredibly rude restaurant staff ...             -0.9884
...                                                   ...                 ...
138365  Wifi terribly slow I speed test network upload...              0.9938
79215   I find anything hotel first I walked past hote...              0.9938
278506  The property great location There bakery next ...              0.9945
339189  Guys I like hotel I wish return next year Howe...              0.9948
480509  I travel lot far visited countless number hote...              0.9957

[515738 rows x 2 columns]
                                     

In [12]:
df = df.reindex(["Hotel_Name", "Hotel_Address", "Total_Number_of_Reviews", "Average_Score", "Reviewer_Score", "Negative_Sentiment", "Positive_Sentiment", "Reviewer_Nationality", "Leisure_trip", "Couple", "Solo_traveler", "Business_trip", "Group", "Family_with_young_children", "Family_with_older_children", "With_a_pet", "Negative_Review", "Positive_Review"], axis=1)

save_csv_file = "../../data/Hotel_Reviews_NLP.csv"
print(f"Saving results to {save_csv_file}")
df.to_csv(save_csv_file, index=False)

Saving results to ../../data/Hotel_Reviews_NLP.csv
